# From Noiseless Samples
We want to interpolate random samples. The thin gray curve gives the traditional linear interpolation, while the thicker blue curve gives the piecewise polynomial spline. The knots of the spline (the places where the pieces of polynomials meet) are shown as black dots and the spline values at the integers are shown as small circles. The interpolating spline is always well-defined when the period is odd; it is also also always well-defined when the delay is not a half integer. However, exact interpolation is not possible when the period is even and the delay becomes a half integer; this instability can be tamed by regularizing the spline, at the cost of an inexact interpolation.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_samples = 15 # Maximal support of the data samples
max_degree = 5 # Maximal spline degree
max_delay = 3.0 # Maximal absolute delay

# Persistent parameters
k0 = -1
n0 = -1
s0 = np.zeros(0)
r0 = sk.interval.Empty()

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Plot
def update_plot (
    period = 6,
    degree = 3,
    delay = 0.0,
    regularized = False
):
    global k0 # Period
    global n0 # Degree
    global s0 # Samples
    global r0 # Display range

    # Random samples
    if period != k0 or degree != n0: # Check for the need of new data
        k0 = period
        n0 = degree
        s0 = rng.standard_normal(period) # Fresh samples
        # Fix a display range
        fn = sk.PeriodicSpline1D.from_samples(s0, degree = n0, delay = delay)
        image = fn.image()
        r0 = sk.interval.Closed((
            min(min(s0), image.infimum) - 0.05,
            max(max(s0), image.supremum) + 0.05
        ))
    # Spline of degree n0 from s0 with current delay and regularization
    fn = sk.PeriodicSpline1D.from_samples(
        s0,
        degree = n0,
        delay = delay,
        regularized = regularized
    )
    # Undelayed linear interpolation of the samples
    f1 = sk.PeriodicSpline1D.from_samples(s0, degree = 1)
    # Plot canvas
    (fig, ax) = plt.subplots()
    # Unadorned linear interpolation in thin gray
    f1.plot(
        (fig, ax),
        plotpoints = 301,
        plotrange = r0,
        curve_fmt = "-C7",
        curve_lw = 0.5,
        curve_markerfmt = " ",
        curvestem_linefmt = "None",
        knot_marker = " ",
        periodbound_markerfmt = " ",
        periodboundstem_linefmt = "None"
    )
    # Spline in default style
    fn.plot((fig, ax), plotpoints = 301)
    plt.show()

widgets.interactive(
    update_plot,
    period = (1, max_samples),
    degree = (0, max_degree),
    delay = (-max_delay, max_delay),
    regularized = widgets.Checkbox(value = False, description = 'Regularized')
)
